In [ ]:
plt.figure(figsize=(12, 5))
sns.histplot(df_new['price'], kde=True)
plt.title('Распределение цены продажи до логарифмирования')
plt.show ()

In [ ]:
sns.histplot(np.log1p(df_new['price']), kde=True)
plt.title('Распределение цены продажи после логарифмирования')
plt.xlabel('Log(Price)')

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(17, 10))
features = ['price', 'sqft_living', 'sqft_lot', 'bedrooms', 'bathrooms', 'price_per_sqft']
for i, feature in enumerate(features):
    row, col = i // 3, i % 3
    sns.boxplot(y=df_new[feature], ax=axes[row, col])
    axes[row, col].set_title(f'Boxplot of {feature}')
plt.show()

In [ ]:
numeric_cols = df_new.select_dtypes(include=[np.number]).columns.tolist()
df_new[numeric_cols].hist(figsize=(16, 20), bins=50)
plt.show()

Корреляции

In [ ]:
exclude_cols = ['id', 'date', 'zipcode', 'lat', 'lon']

In [ ]:
corr = df_new[corr_cols].corrwith(df_new['price']).sort_values(ascending=False)
corr

In [ ]:
plt.figure(figsize=(12, 8))
sns.barplot(y=corr.index, x=corr.values, palette='coolwarm')
plt.title('Корреляция признаков с ценой продажи')
plt.xlabel('Коэффициент корреляции')
plt.show()

In [ ]:
plt.figure(figsize=(12, 8))
top_features = corr.head(10).index.tolist()
sns.heatmap(df_new[top_features + ['price']].corr(), annot=True, fmt='.2f')
plt.title('Матрица корреляций топ-10 признаков')
plt.show()

Зависимость цены от ключевых признаков

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
scatter_features = ['sqft_living', 'sqft_above', 'grade', 'bathrooms', 'bedrooms', 'view']
for i, feature in enumerate(scatter_features):
    row, col = i // 3, i % 3
    axes[row, col].scatter(df_new[feature], df_new['price'], alpha=0.3)
    axes[row, col].set_xlabel(feature)
    axes[row, col].set_ylabel('Price')
    axes[row, col].set_title(f'Price vs {feature}')
plt.show()

Влияние вида на набережную

In [ ]:
plt.figure(figsize=(12, 8))
sns.boxplot(x='waterfront', y='price', data=df_new)
plt.title('Влияние вида на набережную на цену дома')
plt.xlabel('Наличие вида на набережную (0 - нет, 1 - есть)')
plt.ylabel('Price')
plt.show()

In [ ]:
df_new[df_new['waterfront']==1]['price'].median()

In [ ]:
df_new[df_new['waterfront']==0]['price'].median()

Влияние реновации на ценц

In [ ]:
plt.figure(figsize=(10, 6))
sns.boxplot(x='renovated', y='price', data=df_new)
plt.title('Влияние реновации на цену дома')
plt.xlabel('Реновирован (0 - нет, 1 - есть)')
plt.ylabel('Price')
plt.show()

Распределение цен по ZIP-кодам

In [ ]:
plt.figure(figsize=(14, 6))
zipcode_price = df_new.groupby('zipcode')['price'].median().sort_values(ascending=False)
sns.barplot(x=zipcode_price.index, y=zipcode_price.values, palette='coolwarm')
plt.title('Топ-20 почтовых индексов по медианной цене')
plt.xlabel('Zipcode')
plt.ylabel('Медианная цена')
plt.xticks(rotation=90)
plt.show()

Карта распределения цен

In [ ]:
plt.figure(figsize=(12, 8))
scatter = plt.scatter(df_new['long'], df_new['lat'], c=df_new['price'], cmap='viridis', alpha=0.5, s=10)
plt.colorbar(scatter, label='Price')
plt.title('Географическое распределение цен на недвижимость')
plt.xlabel('Долгота')
plt.ylabel('Широта')
plt.show()

Распределение домов по секторам

In [ ]:
def assign_sector(zipcode):
    if zipcode <= 98028:
        return 'A'
    elif zipcode <= 98072:
        return 'B'
    elif zipcode < 98122:
        return 'C'
    else:
        return 'D'

In [ ]:
df_new['sector'] = df_new['zipcode'].apply(assign_sector)
df_new['sector'].value_counts()

In [ ]:
plt.figure(figsize=(10, 6))
sns.boxplot(x='sector', y='price', data=df_new)
plt.title('Распределение цен по секторам')
plt.show()

Временные ряды

In [ ]:
monthly_sales = df_new.groupby('month').size()
monthly_avg_price = df_new.groupby('month')['price'].mean()

In [ ]:
fig, ax1 = plt.subplots(figsize=(12, 6))

ax1.bar(monthly_sales.index, monthly_sales.values, alpha=0.7)
ax1.set_xlabel('Месяц')
ax1.set_ylabel('Количество продаж')
ax2 = ax1.twinx()
ax2.plot(monthly_avg_price.index, monthly_avg_price.values, 'o-')
ax2.set_ylabel('Средняя цена')
plt.title('Сезонность продаж недвижимости')
plt.show()

In [ ]:
df_new.groupby('year')['price'].describe()